# 04 — Modelling Sistem Rekomendasi Film Menggunakan PySpark

Notebook ini digunakan untuk membangun model rekomendasi film berbasis metadata TMDB yang sudah melalui proses scraping, enrichment, EDA, dan preprocessing.

Tujuan utama notebook ini adalah membuat sistem rekomendasi film menggunakan pendekatan *content-based filtering*. Karena dataset yang digunakan tidak memiliki data interaksi pengguna seperti user rating atau histori tontonan, maka sistem rekomendasi dibangun dari kemiripan konten film, yaitu genre, sutradara, aktor, keyword, overview, dan metadata kualitas film.

Model atau pendekatan yang digunakan:

1. **CountVectorizer + Cosine Similarity** sebagai baseline.
2. **TF-IDF + Cosine Similarity** sebagai model utama.
3. **K-Means Clustering** sebagai analisis segmentasi film.
4. **Hybrid Recommendation** sebagai sistem rekomendasi final.

## 1. Alur Modelling

Alur modelling pada notebook ini dirancang agar mudah dijelaskan saat presentasi.

```
Dataset hasil preprocessing
        ↓
Movie document
        ↓
Baseline: CountVectorizer + Cosine Similarity
        ↓
Main model: TF-IDF + Cosine Similarity
        ↓
K-Means Clustering untuk segmentasi film
        ↓
Hybrid Recommendation
        ↓
Evaluasi rekomendasi
        ↓
Output laporan modelling
```

File utama yang digunakan adalah:

- `data/processed/tmdb/tmdb_movie_documents.csv`
- `data/processed/tmdb/tmdb_movies_model_ready.csv`

File `tmdb_movie_documents.csv` berisi teks gabungan dari atribut film. Kolom `movie_document` inilah yang akan diubah menjadi vektor numerik untuk model rekomendasi.

## 2. Setup Library dan SparkSession

Cell ini menyiapkan environment PySpark untuk VSCode di Windows. Karena sebelumnya Spark berhasil berjalan pada path project yang sudah dibersihkan dari tanda kurung, konfigurasi ini mempertahankan setup yang sama.

Catatan:
- `JAVA_HOME` diarahkan ke JDK 17.
- Temp folder Spark diarahkan ke `C:/tmp/spark-temp`.
- Spark dijalankan secara lokal menggunakan `local[2]`.
- Arrow dinonaktifkan agar lebih stabil di Windows.

In [ ]:
import os
import sys
import math
import re
from pathlib import Path

import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.linalg import SparseVector, DenseVector

# Setup Java untuk Windows
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

# Temp folder aman untuk Spark di Windows
Path("C:/tmp/spark-temp").mkdir(parents=True, exist_ok=True)
os.environ["TEMP"] = "C:/tmp/spark-temp"
os.environ["TMP"] = "C:/tmp/spark-temp"

# Pastikan PySpark memakai Python dari venv aktif
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .appName("TMDB Movie Recommendation - Modelling")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Python executable:", sys.executable)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

## 3. Konfigurasi Path Project

Notebook ini diasumsikan berada di folder `notebooks`. Jika dijalankan dari folder `notebooks`, maka `PROJECT_ROOT` akan otomatis naik satu level ke root project.

Input yang digunakan:

- `tmdb_movie_documents.csv`: dataset utama untuk modelling berbasis teks.
- `tmdb_movies_model_ready.csv`: dataset metadata tambahan untuk hybrid scoring.

Output modelling akan disimpan ke:

- `models/recommendation/`
- `reports/modelling/`

In [ ]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "tmdb"
MODEL_DIR = PROJECT_ROOT / "models" / "recommendation"
REPORT_DIR = PROJECT_ROOT / "reports" / "modelling"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

DOCUMENT_PATH = PROCESSED_DIR / "tmdb_movie_documents.csv"
MODEL_READY_PATH = PROCESSED_DIR / "tmdb_movies_model_ready.csv"

RECOMMENDATION_REPORT_OUTPUT = REPORT_DIR / "recommendation_examples.csv"
CLUSTER_REPORT_OUTPUT = REPORT_DIR / "kmeans_cluster_summary.csv"
MODEL_COMPARISON_OUTPUT = REPORT_DIR / "model_comparison_summary.csv"

print("Current dir       :", CURRENT_DIR)
print("Project root      :", PROJECT_ROOT)
print("Document path     :", DOCUMENT_PATH)
print("Document exists   :", DOCUMENT_PATH.exists())
print("Model ready path  :", MODEL_READY_PATH)
print("Model ready exists:", MODEL_READY_PATH.exists())
print("Report dir        :", REPORT_DIR)

## 4. Load Dataset Model

Cell ini membaca dataset hasil preprocessing.

Kolom yang minimal dibutuhkan:

- `id`
- `title`
- `movie_document`

Kolom tambahan seperti `vote_average_clean`, `vote_count_clean`, `popularity_clean`, dan `recommendation_quality_score` akan digunakan pada model hybrid.

In [ ]:
df_docs = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(str(DOCUMENT_PATH))
)

print("Rows dokumen:", df_docs.count())
print("Columns     :", len(df_docs.columns))

df_docs.printSchema()
df_docs.select("id", "title", "movie_document").show(5, truncate=120)

## 5. Validasi dan Persiapan Data

Pada tahap ini dilakukan validasi kolom wajib, pembersihan nilai kosong, dan deduplikasi berdasarkan ID film.

Data yang tidak memiliki `movie_document` tidak digunakan karena model rekomendasi membutuhkan representasi teks dari film.

In [ ]:
required_cols = ["id", "title", "movie_document"]
missing_cols = [c for c in required_cols if c not in df_docs.columns]

if missing_cols:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_cols}")

df_base = (
    df_docs
    .withColumn("id", F.col("id").cast("int"))
    .withColumn("title", F.coalesce(F.col("title").cast("string"), F.lit("")))
    .withColumn("movie_document", F.coalesce(F.col("movie_document").cast("string"), F.lit("")))
    .filter(F.col("id").isNotNull())
    .filter(F.length(F.trim(F.col("movie_document"))) > 0)
    .dropDuplicates(["id"])
)

# Kolom numerik tambahan untuk hybrid model
numeric_defaults = {
    "vote_average_clean": 0.0,
    "vote_count_clean": 0.0,
    "popularity_clean": 0.0,
    "runtime_clean": 0.0,
    "recommendation_quality_score": 0.0,
}

for c, default_value in numeric_defaults.items():
    if c not in df_base.columns:
        df_base = df_base.withColumn(c, F.lit(default_value))
    else:
        df_base = df_base.withColumn(c, F.coalesce(F.col(c).cast("double"), F.lit(default_value)))

if "release_year" not in df_base.columns:
    df_base = df_base.withColumn("release_year", F.lit(None).cast("int"))
else:
    df_base = df_base.withColumn("release_year", F.col("release_year").cast("int"))

if "genres_model" not in df_base.columns:
    df_base = df_base.withColumn("genres_model", F.lit(""))
else:
    df_base = df_base.withColumn("genres_model", F.coalesce(F.col("genres_model").cast("string"), F.lit("")))

if "director_model" not in df_base.columns:
    df_base = df_base.withColumn("director_model", F.lit(""))
else:
    df_base = df_base.withColumn("director_model", F.coalesce(F.col("director_model").cast("string"), F.lit("")))

df_base = df_base.cache()

print("Rows setelah validasi:", df_base.count())
print("Duplicate id:", df_base.count() - df_base.select("id").distinct().count())

df_base.select(
    "id", "title", "release_year", "vote_average_clean",
    "vote_count_clean", "popularity_clean", "recommendation_quality_score"
).show(10, truncate=False)

## 6. Helper Function: Cosine Similarity

Cosine Similarity digunakan untuk menghitung kedekatan antarfilm berdasarkan vektor fitur.

Nilai cosine similarity berada pada rentang:

- Mendekati `1`: sangat mirip.
- Mendekati `0`: tidak terlalu mirip.
- Mendekati `-1`: berlawanan arah, namun pada fitur teks non-negatif kasus ini jarang muncul.

Pada sistem rekomendasi ini, film dengan similarity tertinggi akan menjadi kandidat rekomendasi.

In [ ]:
def vector_norm(v):
    if v is None:
        return 0.0
    if isinstance(v, SparseVector):
        return float(math.sqrt(sum(x * x for x in v.values)))
    if isinstance(v, DenseVector):
        return float(math.sqrt(sum(x * x for x in v)))
    arr = np.array(v)
    return float(np.linalg.norm(arr))

def vector_dot(v1, v2):
    if v1 is None or v2 is None:
        return 0.0

    if isinstance(v1, SparseVector) and isinstance(v2, SparseVector):
        d1 = dict(zip(v1.indices, v1.values))
        d2 = dict(zip(v2.indices, v2.values))
        if len(d1) > len(d2):
            d1, d2 = d2, d1
        return float(sum(value * d2.get(idx, 0.0) for idx, value in d1.items()))

    if isinstance(v1, SparseVector):
        return float(sum(value * v2[int(idx)] for idx, value in zip(v1.indices, v1.values)))

    if isinstance(v2, SparseVector):
        return float(sum(value * v1[int(idx)] for idx, value in zip(v2.indices, v2.values)))

    return float(np.dot(np.array(v1), np.array(v2)))

def cosine_similarity_vector(v1, v2):
    norm1 = vector_norm(v1)
    norm2 = vector_norm(v2)
    if norm1 == 0.0 or norm2 == 0.0:
        return 0.0
    return float(vector_dot(v1, v2) / (norm1 * norm2))

def find_movie_by_title(dataframe, title_keyword):
    keyword = title_keyword.lower().strip()
    result = (
        dataframe
        .filter(F.lower(F.col("title")).contains(keyword))
        .orderBy(F.desc("vote_count_clean"), F.desc("popularity_clean"))
        .limit(1)
        .collect()
    )

    if not result:
        raise ValueError(f"Film dengan keyword judul '{title_keyword}' tidak ditemukan.")

    return result[0]

print("Helper cosine similarity siap digunakan.")

# Model 1 — CountVectorizer + Cosine Similarity

Model pertama digunakan sebagai baseline. CountVectorizer mengubah teks menjadi vektor berdasarkan frekuensi kemunculan kata.

Kelebihan:
- Sederhana.
- Mudah dijelaskan.
- Cocok sebagai pembanding model utama.

Kekurangan:
- Tidak membedakan kata umum dan kata spesifik.
- Kata yang sering muncul bisa terlalu dominan.

## 7. Training Baseline CountVectorizer

Pipeline baseline:

```
movie_document
        ↓
RegexTokenizer
        ↓
StopWordsRemover
        ↓
CountVectorizer
        ↓
count_features
```

Kolom `count_features` kemudian dipakai untuk menghitung cosine similarity.

In [ ]:
regex_tokenizer_cv = RegexTokenizer(
    inputCol="movie_document",
    outputCol="tokens_cv",
    pattern="\\W+",
    minTokenLength=2
)

stopwords_remover_cv = StopWordsRemover(
    inputCol="tokens_cv",
    outputCol="filtered_tokens_cv"
)

count_vectorizer = CountVectorizer(
    inputCol="filtered_tokens_cv",
    outputCol="count_features",
    vocabSize=50000,
    minDF=3.0
)

pipeline_cv = Pipeline(stages=[
    regex_tokenizer_cv,
    stopwords_remover_cv,
    count_vectorizer
])

cv_model = pipeline_cv.fit(df_base)
df_cv = cv_model.transform(df_base).cache()

print("CountVectorizer model selesai.")
print("Rows:", df_cv.count())

df_cv.select("id", "title", "count_features").show(5, truncate=80)

## 8. Fungsi Rekomendasi Baseline CountVectorizer

Fungsi ini menerima keyword judul film, mencari film yang cocok, lalu menghitung cosine similarity terhadap seluruh film lain.

In [ ]:
def recommend_countvectorizer(title_keyword, top_n=10):
    query_row = find_movie_by_title(df_cv, title_keyword)
    query_id = query_row["id"]
    query_title = query_row["title"]
    query_vector = query_row["count_features"]

    cosine_udf = F.udf(lambda v: float(cosine_similarity_vector(v, query_vector)), T.DoubleType())

    recs = (
        df_cv
        .withColumn("similarity_score", cosine_udf(F.col("count_features")))
        .filter(F.col("id") != query_id)
        .orderBy(F.desc("similarity_score"), F.desc("vote_count_clean"))
        .select(
            "id", "title", "release_year", "genres_model", "director_model",
            "vote_average_clean", "vote_count_clean", "popularity_clean",
            "similarity_score"
        )
        .limit(top_n)
    )

    print(f"Query movie: {query_title} (id={query_id})")
    return recs

# Contoh penggunaan
recommend_countvectorizer("interstellar", top_n=10).show(10, truncate=False)

# Model 2 — TF-IDF + Cosine Similarity

Model kedua adalah model utama. TF-IDF memberi bobot lebih tinggi pada kata yang lebih spesifik dan menurunkan bobot kata yang terlalu umum.

Kelebihan:
- Lebih baik dibanding CountVectorizer untuk fitur teks.
- Cocok untuk content-based filtering.
- Mudah dijelaskan secara akademik.

Pada project ini, model TF-IDF digunakan sebagai dasar utama untuk menghitung kemiripan antarfilm.

## 9. Training TF-IDF Model

Pipeline TF-IDF:

```
movie_document
        ↓
RegexTokenizer
        ↓
StopWordsRemover
        ↓
CountVectorizer
        ↓
IDF
        ↓
tfidf_features
```

In [ ]:
regex_tokenizer_tfidf = RegexTokenizer(
    inputCol="movie_document",
    outputCol="tokens_tfidf",
    pattern="\\W+",
    minTokenLength=2
)

stopwords_remover_tfidf = StopWordsRemover(
    inputCol="tokens_tfidf",
    outputCol="filtered_tokens_tfidf"
)

tf_count_vectorizer = CountVectorizer(
    inputCol="filtered_tokens_tfidf",
    outputCol="tf_raw_features",
    vocabSize=60000,
    minDF=3.0
)

idf = IDF(
    inputCol="tf_raw_features",
    outputCol="tfidf_features"
)

pipeline_tfidf = Pipeline(stages=[
    regex_tokenizer_tfidf,
    stopwords_remover_tfidf,
    tf_count_vectorizer,
    idf
])

tfidf_model = pipeline_tfidf.fit(df_base)
df_tfidf = tfidf_model.transform(df_base).cache()

print("TF-IDF model selesai.")
print("Rows:", df_tfidf.count())

df_tfidf.select("id", "title", "tfidf_features").show(5, truncate=80)

## 10. Fungsi Rekomendasi TF-IDF

Fungsi ini digunakan sebagai model utama rekomendasi. Cara kerjanya sama seperti baseline, tetapi fitur yang digunakan adalah `tfidf_features`.

In [ ]:
def recommend_tfidf(title_keyword, top_n=10):
    query_row = find_movie_by_title(df_tfidf, title_keyword)
    query_id = query_row["id"]
    query_title = query_row["title"]
    query_vector = query_row["tfidf_features"]

    cosine_udf = F.udf(lambda v: float(cosine_similarity_vector(v, query_vector)), T.DoubleType())

    recs = (
        df_tfidf
        .withColumn("similarity_score", cosine_udf(F.col("tfidf_features")))
        .filter(F.col("id") != query_id)
        .orderBy(F.desc("similarity_score"), F.desc("vote_count_clean"))
        .select(
            "id", "title", "release_year", "genres_model", "director_model",
            "vote_average_clean", "vote_count_clean", "popularity_clean",
            "similarity_score"
        )
        .limit(top_n)
    )

    print(f"Query movie: {query_title} (id={query_id})")
    return recs

recommend_tfidf("interstellar", top_n=10).show(10, truncate=False)

# Model 3 — K-Means Clustering

K-Means digunakan untuk mengelompokkan film berdasarkan fitur TF-IDF.

Pada project ini, K-Means bukan model utama rekomendasi, melainkan model tambahan untuk analisis segmentasi film.

Fungsi K-Means:

- Mengetahui kelompok film yang mirip.
- Membantu memahami struktur dataset.
- Bisa digunakan sebagai filter tambahan pada sistem rekomendasi.

## 11. Training K-Means Clustering

Jumlah cluster awal diset ke `20`. Angka ini bisa diubah sesuai hasil eksperimen.

Untuk dataset film yang besar dan beragam, cluster 10–30 biasanya cukup masuk akal untuk percobaan awal.

In [ ]:
K = 20

kmeans = KMeans(
    featuresCol="tfidf_features",
    predictionCol="cluster_id",
    k=K,
    seed=42,
    maxIter=20
)

kmeans_model = kmeans.fit(df_tfidf)
df_clustered = kmeans_model.transform(df_tfidf).cache()

evaluator = ClusteringEvaluator(
    featuresCol="tfidf_features",
    predictionCol="cluster_id",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

silhouette_score = evaluator.evaluate(df_clustered)

print("K-Means selesai.")
print("Jumlah cluster:", K)
print("Silhouette score:", silhouette_score)

df_clustered.select("id", "title", "cluster_id").show(10, truncate=False)

## 12. Ringkasan Cluster

Cell ini menampilkan jumlah film pada setiap cluster, rata-rata rating, rata-rata vote count, dan rata-rata popularity.

In [ ]:
cluster_summary = (
    df_clustered
    .groupBy("cluster_id")
    .agg(
        F.count("*").alias("movie_count"),
        F.round(F.avg("vote_average_clean"), 3).alias("avg_vote_average"),
        F.round(F.avg("vote_count_clean"), 3).alias("avg_vote_count"),
        F.round(F.avg("popularity_clean"), 3).alias("avg_popularity"),
        F.round(F.avg("recommendation_quality_score"), 3).alias("avg_quality_score")
    )
    .orderBy("cluster_id")
)

cluster_summary.show(K, truncate=False)

## 13. Interpretasi Top Terms per Cluster

Cell ini mencoba membaca kata-kata paling dominan pada masing-masing cluster berdasarkan centroid K-Means.

Top terms membantu kita memahami karakteristik cluster, misalnya cluster action, romance, horror, animation, dan sebagainya.

In [ ]:
# Ambil vocabulary dari CountVectorizer di pipeline TF-IDF
tfidf_cv_model = tfidf_model.stages[2]
vocabulary = tfidf_cv_model.vocabulary

centers = kmeans_model.clusterCenters()

cluster_terms = []

for cluster_idx, center in enumerate(centers):
    arr = center.toArray() if hasattr(center, "toArray") else np.array(center)
    top_indices = arr.argsort()[-15:][::-1]
    top_terms = [vocabulary[i] for i in top_indices if i < len(vocabulary)]
    cluster_terms.append((cluster_idx, " | ".join(top_terms)))

cluster_terms_df = spark.createDataFrame(cluster_terms, ["cluster_id", "top_terms"])
cluster_terms_df.show(K, truncate=False)

# Model 4 — Hybrid Recommendation

Hybrid recommendation adalah sistem final yang menggabungkan:

1. Kemiripan konten dari TF-IDF.
2. Rating film.
3. Jumlah vote.
4. Popularity.
5. Quality score hasil preprocessing.
6. Opsional: cluster yang sama.

Tujuannya agar rekomendasi tidak hanya mirip secara teks, tetapi juga lebih layak direkomendasikan secara kualitas data.

## 14. Persiapan Normalisasi Metadata

Sebelum menghitung hybrid score, beberapa kolom numerik perlu dinormalisasi agar skala nilainya lebih seimbang.

In [ ]:
stats = df_clustered.agg(
    F.max(F.log1p(F.col("vote_count_clean"))).alias("max_log_vote_count"),
    F.max(F.log1p(F.col("popularity_clean"))).alias("max_log_popularity"),
    F.max(F.col("recommendation_quality_score")).alias("max_quality_score")
).collect()[0]

MAX_LOG_VOTE_COUNT = float(stats["max_log_vote_count"] or 1.0)
MAX_LOG_POPULARITY = float(stats["max_log_popularity"] or 1.0)
MAX_QUALITY_SCORE = float(stats["max_quality_score"] or 1.0)

print("MAX_LOG_VOTE_COUNT:", MAX_LOG_VOTE_COUNT)
print("MAX_LOG_POPULARITY:", MAX_LOG_POPULARITY)
print("MAX_QUALITY_SCORE :", MAX_QUALITY_SCORE)

## 15. Fungsi Hybrid Recommendation

Formula hybrid yang digunakan:

```
final_score =
0.70 × similarity_score
+ 0.10 × rating_norm
+ 0.10 × vote_count_norm
+ 0.05 × popularity_norm
+ 0.05 × quality_score_norm
```

Bobot terbesar tetap pada similarity karena sistem ini adalah content-based recommendation. Metadata hanya digunakan sebagai penguat kualitas rekomendasi.

In [ ]:
def recommend_hybrid(title_keyword, top_n=10, same_cluster_only=False):
    query_row = find_movie_by_title(df_clustered, title_keyword)
    query_id = query_row["id"]
    query_title = query_row["title"]
    query_vector = query_row["tfidf_features"]
    query_cluster = query_row["cluster_id"]

    cosine_udf = F.udf(lambda v: float(cosine_similarity_vector(v, query_vector)), T.DoubleType())

    candidates = (
        df_clustered
        .withColumn("similarity_score", cosine_udf(F.col("tfidf_features")))
        .filter(F.col("id") != query_id)
    )

    if same_cluster_only:
        candidates = candidates.filter(F.col("cluster_id") == query_cluster)

    candidates = (
        candidates
        .withColumn("rating_norm", F.col("vote_average_clean") / F.lit(10.0))
        .withColumn("vote_count_norm", F.log1p(F.col("vote_count_clean")) / F.lit(MAX_LOG_VOTE_COUNT))
        .withColumn("popularity_norm", F.log1p(F.col("popularity_clean")) / F.lit(MAX_LOG_POPULARITY))
        .withColumn("quality_norm", F.col("recommendation_quality_score") / F.lit(MAX_QUALITY_SCORE))
        .withColumn(
            "hybrid_score",
            (F.col("similarity_score") * F.lit(0.70)) +
            (F.col("rating_norm") * F.lit(0.10)) +
            (F.col("vote_count_norm") * F.lit(0.10)) +
            (F.col("popularity_norm") * F.lit(0.05)) +
            (F.col("quality_norm") * F.lit(0.05))
        )
    )

    recs = (
        candidates
        .orderBy(F.desc("hybrid_score"), F.desc("similarity_score"), F.desc("vote_count_clean"))
        .select(
            "id", "title", "release_year", "genres_model", "director_model",
            "cluster_id", "vote_average_clean", "vote_count_clean", "popularity_clean",
            "similarity_score", "rating_norm", "vote_count_norm",
            "popularity_norm", "quality_norm", "hybrid_score"
        )
        .limit(top_n)
    )

    print(f"Query movie: {query_title} (id={query_id}, cluster={query_cluster})")
    return recs

recommend_hybrid("interstellar", top_n=10, same_cluster_only=False).show(10, truncate=False)

## 16. Hybrid Recommendation dengan Filter Cluster

Pada variasi ini, sistem hanya mengambil kandidat film dari cluster yang sama dengan film input.

Kegunaannya:
- Membatasi rekomendasi agar tetap berada pada kelompok film yang mirip.
- Mengurangi rekomendasi yang terlalu jauh dari konteks film awal.

In [ ]:
recommend_hybrid("interstellar", top_n=10, same_cluster_only=True).show(10, truncate=False)

# Evaluasi Rekomendasi

Dataset TMDB yang digunakan tidak memiliki data interaksi pengguna. Karena itu, evaluasi tidak menggunakan metrik collaborative filtering seperti RMSE atau MAE.

Evaluasi yang digunakan bersifat content-based, yaitu:

1. Rata-rata similarity score.
2. Rata-rata rating film rekomendasi.
3. Rata-rata vote count film rekomendasi.
4. Rata-rata popularity film rekomendasi.
5. Analisis kualitatif terhadap hasil Top-N rekomendasi.

## 17. Test Case Rekomendasi

Cell ini menjalankan beberapa contoh film untuk membandingkan hasil rekomendasi dari:

- CountVectorizer baseline.
- TF-IDF main model.
- Hybrid final model.

In [ ]:
test_queries = [
    "interstellar",
    "toy story",
    "the dark knight",
    "avatar",
    "titanic"
]

def collect_recommendation_metrics(model_name, query, rec_df, score_col):
    pdf = rec_df.toPandas()

    if len(pdf) == 0:
        return {
            "model": model_name,
            "query": query,
            "recommendation_count": 0,
            "avg_score": 0.0,
            "avg_rating": 0.0,
            "avg_vote_count": 0.0,
            "avg_popularity": 0.0,
        }, pdf

    return {
        "model": model_name,
        "query": query,
        "recommendation_count": len(pdf),
        "avg_score": float(pdf[score_col].mean()),
        "avg_rating": float(pdf["vote_average_clean"].mean()),
        "avg_vote_count": float(pdf["vote_count_clean"].mean()),
        "avg_popularity": float(pdf["popularity_clean"].mean()),
    }, pdf

all_metrics = []
all_recommendations = []

for query in test_queries:
    print("=" * 100)
    print("QUERY:", query)

    try:
        rec_cv = recommend_countvectorizer(query, top_n=10)
        metric_cv, pdf_cv = collect_recommendation_metrics("CountVectorizer + Cosine", query, rec_cv, "similarity_score")
        all_metrics.append(metric_cv)
        pdf_cv["model"] = "CountVectorizer + Cosine"
        pdf_cv["query"] = query
        all_recommendations.append(pdf_cv)

        rec_tfidf = recommend_tfidf(query, top_n=10)
        metric_tfidf, pdf_tfidf = collect_recommendation_metrics("TF-IDF + Cosine", query, rec_tfidf, "similarity_score")
        all_metrics.append(metric_tfidf)
        pdf_tfidf["model"] = "TF-IDF + Cosine"
        pdf_tfidf["query"] = query
        all_recommendations.append(pdf_tfidf)

        rec_hybrid = recommend_hybrid(query, top_n=10, same_cluster_only=False)
        metric_hybrid, pdf_hybrid = collect_recommendation_metrics("Hybrid Recommendation", query, rec_hybrid, "hybrid_score")
        all_metrics.append(metric_hybrid)
        pdf_hybrid["model"] = "Hybrid Recommendation"
        pdf_hybrid["query"] = query
        all_recommendations.append(pdf_hybrid)

    except Exception as e:
        print("Gagal untuk query:", query, "| Error:", e)

metrics_pdf = pd.DataFrame(all_metrics)
recommendations_pdf = pd.concat(all_recommendations, ignore_index=True) if all_recommendations else pd.DataFrame()

metrics_pdf

## 18. Ringkasan Perbandingan Model

Cell ini menampilkan ringkasan metrik dari beberapa test case. Metrik ini bukan evaluasi ground truth, tetapi digunakan untuk membandingkan karakteristik output masing-masing pendekatan.

In [ ]:
metrics_pdf

## 19. Simpan Laporan Modelling

Karena Spark lokal di Windows dapat bermasalah saat menulis Parquet atau format native Hadoop, output laporan disimpan menggunakan Pandas CSV.

File yang disimpan:

- `recommendation_examples.csv`
- `model_comparison_summary.csv`
- `kmeans_cluster_summary.csv`

In [ ]:
# Simpan rekomendasi contoh
if len(recommendations_pdf) > 0:
    recommendations_pdf.to_csv(RECOMMENDATION_REPORT_OUTPUT, index=False)

# Simpan ringkasan model
metrics_pdf.to_csv(MODEL_COMPARISON_OUTPUT, index=False)

# Simpan ringkasan cluster
cluster_summary_pdf = cluster_summary.toPandas()
cluster_terms_pdf = cluster_terms_df.toPandas()

cluster_report_pdf = cluster_summary_pdf.merge(cluster_terms_pdf, on="cluster_id", how="left")
cluster_report_pdf.to_csv(CLUSTER_REPORT_OUTPUT, index=False)

print("Saved recommendation examples:", RECOMMENDATION_REPORT_OUTPUT)
print("Saved model comparison       :", MODEL_COMPARISON_OUTPUT)
print("Saved cluster summary        :", CLUSTER_REPORT_OUTPUT)

# Kesimpulan Modelling

Berdasarkan rancangan model pada notebook ini:

1. **CountVectorizer + Cosine Similarity** digunakan sebagai baseline karena sederhana dan mudah dibandingkan.
2. **TF-IDF + Cosine Similarity** digunakan sebagai model utama karena mampu memberi bobot lebih baik pada kata yang spesifik.
3. **K-Means Clustering** digunakan untuk segmentasi film dan analisis pola kelompok film.
4. **Hybrid Recommendation** digunakan sebagai sistem final karena menggabungkan kemiripan konten dengan rating, vote count, popularity, dan quality score.

Model final yang direkomendasikan untuk sistem adalah **Hybrid Recommendation berbasis TF-IDF**, karena hasilnya tidak hanya mempertimbangkan kemiripan konten, tetapi juga kualitas metadata film.

## Narasi Singkat untuk Presentasi

Model rekomendasi yang digunakan dalam proyek ini adalah content-based filtering. Dataset TMDB tidak memiliki data interaksi pengguna, sehingga pendekatan collaborative filtering seperti ALS tidak digunakan. Model utama dibangun menggunakan TF-IDF dan Cosine Similarity untuk menghitung kemiripan antarfilm berdasarkan metadata seperti genre, aktor, sutradara, keyword, dan overview. Sebagai pembanding, dibuat baseline menggunakan CountVectorizer. Selain itu, K-Means digunakan untuk segmentasi film, sedangkan model final menggunakan pendekatan hybrid dengan menambahkan rating, vote count, popularity, dan quality score agar rekomendasi yang diberikan lebih relevan dan layak ditonton.